In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Escribe un pase de transpilador personalizado


<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o versiones más recientes.

```
qiskit[all]~=2.3.0
```
</details>
El SDK de Qiskit te permite crear pases de transpilación personalizados y ejecutarlos en el objeto `PassManager` o agregarlos a un `StagedPassManager`. Aquí demostraremos cómo escribir un pase de transpilador, con énfasis en construir un pase que aplique [Pauli twirling](https://arxiv.org/abs/quant-ph/0606161) sobre las compuertas cuánticas ruidosas de un circuito cuántico. Este ejemplo usa el DAG, que es el objeto manipulado por el tipo de pase `TransformationPass`.

<details>
  <summary>
    Contexto: representación DAG
  </summary>

Antes de construir un pase, es importante presentar la representación interna de los circuitos cuánticos en Qiskit: el [grafo acíclico dirigido (DAG)](../api/qiskit/qiskit.dagcircuit.DAGCircuit) (consulta [este tutorial](https://qiskit.org/ecosystem/rustworkx/tutorial/dags.html) para obtener una introducción). Para seguir estos pasos, instala la [biblioteca `graphviz`](https://graphviz.org/download/) para las funciones de visualización del DAG.

En Qiskit, dentro de las etapas de transpilación, los circuitos se representan mediante un DAG. En general, un DAG está compuesto por *vértices* (también llamados "nodos") y *aristas* dirigidas que conectan pares de vértices en una orientación específica. Esta representación se almacena usando objetos `qiskit.dagcircuit.DAGCircuit` compuestos de objetos individuales `DagNode`. La ventaja de esta representación frente a una lista pura de compuertas (es decir, un *netlist*) es que el flujo de información entre operaciones es explícito, lo que facilita la toma de decisiones de transformación.

Este ejemplo ilustra el DAG creando un circuito sencillo que prepara un estado de Bell y aplica una rotación $R_Z$ dependiendo del resultado de la medición.

In [1]:
from qiskit.dagcircuit import DAGCircuit
from qiskit.circuit import QuantumCircuit, QuantumRegister, Gate
from qiskit.circuit.library import CXGate, ECRGate
from qiskit.transpiler import PassManager
from qiskit.transpiler.basepasses import TransformationPass
from qiskit.quantum_info import Operator, pauli_basis

import numpy as np

from typing import Iterable, Optional

In [2]:
class PauliTwirl(TransformationPass):
    """Add Pauli twirls to two-qubit gates."""

    def __init__(
        self,
        gates_to_twirl: Optional[Iterable[Gate]] = None,
    ):
        """
        Args:
            gates_to_twirl: Names of gates to twirl. The default behavior is to twirl all
                two-qubit basis gates, `cx` and `ecr` for IBM backends.
        """
        if gates_to_twirl is None:
            gates_to_twirl = [CXGate(), ECRGate()]
        self.gates_to_twirl = gates_to_twirl
        self.build_twirl_set()
        super().__init__()

    def build_twirl_set(self):
        """
        Build a set of Paulis to twirl for each gate and store internally as .twirl_set.
        """
        self.twirl_set = {}

        # iterate through gates to be twirled
        for twirl_gate in self.gates_to_twirl:
            twirl_list = []

            # iterate through Paulis on left of gate to twirl
            for pauli_left in pauli_basis(2):
                # iterate through Paulis on right of gate to twirl
                for pauli_right in pauli_basis(2):
                    # save pairs that produce identical operation as gate to twirl
                    if (Operator(pauli_left) @ Operator(twirl_gate)).equiv(
                        Operator(twirl_gate) @ pauli_right
                    ):
                        twirl_list.append((pauli_left, pauli_right))

            self.twirl_set[twirl_gate.name] = twirl_list

    def run(
        self,
        dag: DAGCircuit,
    ) -> DAGCircuit:
        # collect all nodes in DAG and proceed if it is to be twirled
        twirling_gate_classes = tuple(
            gate.base_class for gate in self.gates_to_twirl
        )
        for node in dag.op_nodes():
            if not isinstance(node.op, twirling_gate_classes):
                continue

            # random integer to select Pauli twirl pair
            pauli_index = np.random.randint(
                0, len(self.twirl_set[node.op.name])
            )
            twirl_pair = self.twirl_set[node.op.name][pauli_index]

            # instantiate mini_dag and attach quantum register
            mini_dag = DAGCircuit()
            register = QuantumRegister(2)
            mini_dag.add_qreg(register)

            # apply left Pauli, gate to twirl, and right Pauli to empty mini-DAG
            mini_dag.apply_operation_back(
                twirl_pair[0].to_instruction(), [register[0], register[1]]
            )
            mini_dag.apply_operation_back(node.op, [register[0], register[1]])
            mini_dag.apply_operation_back(
                twirl_pair[1].to_instruction(), [register[0], register[1]]
            )

            # substitute gate to twirl node with twirling mini-DAG
            dag.substitute_node_with_dag(node, mini_dag)

        return dag

![El DAG del circuito está formado por nodos conectados mediante aristas direccionales. Es una forma visual de representar los qubits o bits clásicos, las operaciones y el flujo de datos.](../docs/images/guides/custom-transpiler-pass/DAG.avif "DAG")
</details>
## Pases de transpilador
Los pases de transpilador se clasifican como [`AnalysisPass`](../api/qiskit/qiskit.transpiler.AnalysisPass) o [`TransformationPass`](../api/qiskit/qiskit.transpiler.TransformationPass). En general, los pases trabajan con el [DAG](../api/qiskit/qiskit.dagcircuit.DAGCircuit) y el `property_set`, un objeto similar a un diccionario para almacenar propiedades determinadas por los pases de análisis. Los pases de análisis trabajan tanto con el DAG como con su `property_set`: no pueden modificar el DAG, pero sí pueden modificar el `property_set`. Esto contrasta con los pases de transformación, que sí modifican el DAG y pueden leer (pero no escribir en) el `property_set`. Por ejemplo, los pases de transformación traducen un circuito a su [ISA](./transpile#instruction-set-architecture) o realizan pases de enrutamiento para insertar compuertas SWAP donde sea necesario.
## Crea un pase de transpilador `PauliTwirl`
El siguiente ejemplo construye un pase de transpilador que agrega Pauli twirls. El [Pauli twirling](https://arxiv.org/abs/quant-ph/0606161) es una estrategia de supresión de errores que aleatoriza cómo los qubits experimentan los canales ruidosos, que en este ejemplo asumimos que son las compuertas de dos qubits (ya que son mucho más propensas a errores que las compuertas de un solo qubit). Los Pauli twirls no afectan la operación de las compuertas de dos qubits. Se eligen de manera que los aplicados *antes* de la compuerta de dos qubits (a la izquierda) sean contrarrestados por los aplicados *después* (a la derecha). En este sentido, las operaciones de dos qubits son idénticas, pero la forma en que se ejecutan es diferente. Una ventaja del Pauli twirling es que convierte los errores coherentes en errores estocásticos, que pueden reducirse promediando más ejecuciones.

Los pases de transpilador actúan sobre el [DAG](../api/qiskit/qiskit.dagcircuit.DAGCircuit), por lo que el método importante a sobrescribir es `.run()`, que recibe el DAG como entrada. Inicializar pares de Paulis como se muestra preserva la operación de cada compuerta de dos qubits. Esto se hace con el método auxiliar `build_twirl_set`, que recorre cada Pauli de dos qubits (obtenido de `pauli_basis(2)`) y encuentra el otro Pauli que preserva la operación.

Desde el DAG, usa el método `op_nodes()` para obtener todos sus nodos. El DAG también puede usarse para recopilar ejecuciones, que son secuencias de nodos que se ejecutan sin interrupciones en un qubit. Estas pueden recopilarse como ejecuciones de un qubit con `collect_1q_runs`, ejecuciones de dos qubits con `collect_2q_runs`, y ejecuciones de nodos cuyos nombres de instrucción estén en una lista de nombres con `collect_runs`. El `DAGCircuit` tiene muchos métodos para buscar y recorrer un grafo. Uno de los más utilizados es `topological_op_nodes`, que proporciona los nodos en orden de dependencia. Otros métodos como `bfs_successors` se usan principalmente para determinar cómo los nodos interactúan con las operaciones posteriores en un DAG.

En el ejemplo, queremos reemplazar cada nodo, que representa una instrucción, con un subcircuito construido como un mini DAG. El mini DAG tiene un registro cuántico de dos qubits agregado. Las operaciones se agregan al mini DAG usando `apply_operation_back`, que coloca la `Instruction` en la salida del mini DAG (mientras que `apply_operation_front` la colocaría en la entrada). Luego el nodo es sustituido por el mini DAG usando `substitute_node_with_dag`, y el proceso continúa sobre cada instancia de `CXGate` y `ECRGate` en el DAG (correspondientes a las compuertas base de dos qubits en los backends de IBM&reg;).

In [3]:
qc = QuantumCircuit(3)
qc.cx(0, 1)
qc.ecr(1, 2)
qc.ecr(1, 0)
qc.cx(2, 1)
qc.draw("mpl")

<Image src="../docs/images/guides/custom-transpiler-pass/extracted-outputs/9123905d-b4cb-4ae9-9695-4ad77e70bdab-0.svg" alt="Output of the previous code cell" />

To apply the custom pass, build a pass manager using the `PauliTwirl` pass and run it on 50 circuits.

In [4]:
pm = PassManager([PauliTwirl()])
twirled_qcs = [pm.run(qc) for _ in range(50)]

## Usa el pase de transpilador `PauliTwirl`
El siguiente código usa el pase creado anteriormente para transpilar un circuito. Considera un circuito sencillo con compuertas `cx` y `ecr`.

In [5]:
twirled_qcs[-1].draw("mpl")

<Image src="../docs/images/guides/custom-transpiler-pass/extracted-outputs/e2515cf3-f8d9-4281-9673-d5a955d7aab9-0.svg" alt="Output of the previous code cell" />

![Salida de la celda de código anterior](../docs/images/guides/custom-transpiler-pass/extracted-outputs/9123905d-b4cb-4ae9-9695-4ad77e70bdab-0.svg)

Para aplicar el pase personalizado, construye un gestor de pases usando el pase `PauliTwirl` y ejecútalo sobre 50 circuitos.

In [6]:
np.all([Operator(twirled_qc).equiv(qc) for twirled_qc in twirled_qcs])

np.True_

Cada compuerta de dos qubits queda ahora flanqueada por dos Paulis.